Preprocess chicken cardiogenesis scRNA-seq data for CPM.

Follows the CPM paper (Bhasker et al. 2025), Section 2.5:
"The dataset was processed following the approach described in Mantri et al. (2021),
with the exception that CPM operated on the full dataset containing 18,797 genes
prior to Scanorama integration and the selection of the top 2000 highly variable genes."

Mantri's preprocessing = Seurat NormalizeData (log-normalize to 10k) + min.cells=3 gene filter.
CPM skips: Scanorama integration, HVG selection, PCA.

Inputs:
  - Seven .h5 files from GEO GSE149457 (scRNA-seq only, GSM4502475-GSM4502481)
  - Mantri's cell metadata CSV from https://github.com/madhavmantri/chicken_heart/tree/master/data

Output:
  - chicken_cardio.csv — ready for make_dataset in data.py

In [30]:
import scanpy as sc
import anndata as ad
import pandas as pd

In [31]:
H5_FILES = {
    'D4':     'chicken_cardio/GSM4502475_chicken_heart_scRNAseq_D4_filtered_feature_bc_matrix.h5',
    'D7-LV':  'chicken_cardio/GSM4502476_chicken_heart_scRNAseq_D7_LV_filtered_feature_bc_matrix.h5',
    'D7-RV':  'chicken_cardio/GSM4502477_chicken_heart_scRNAseq_D7_RV_filtered_feature_bc_matrix.h5',
    'D10-LV': 'chicken_cardio/GSM4502478_chicken_heart_scRNAseq_D10_LV_filtered_feature_bc_matrix.h5',
    'D10-RV': 'chicken_cardio/GSM4502479_chicken_heart_scRNAseq_D10_RV_filtered_feature_bc_matrix.h5',
    'D14-LV': 'chicken_cardio/GSM4502480_chicken_heart_scRNAseq_D14_LV_filtered_feature_bc_matrix.h5',
    'D14-RV': 'chicken_cardio/GSM4502481_chicken_heart_scRNAseq_D14_RV_filtered_feature_bc_matrix.h5',
}
METADATA_PATH = 'chicken_cardio/scRNAseq_metadata.csv'
OUTPUT_CSV = 'chicken_cardio/chicken_cardio.csv'

In [32]:
adatas = []
for sample, path in H5_FILES.items():
    a = sc.read_10x_h5(path)
    a.var_names_make_unique()
    a.obs['sample'] = sample
    a.obs_names = [f'{sample}_{bc.split("-")[0]}' for bc in a.obs_names]
    adatas.append(a)

adata = ad.concat(adatas, join='outer', merge='same')
print(f'Raw combined: {adata.shape}')

C:\Users\natal\Desktop\UBC\OG_CPM\code\.venv\lib\site-packages\anndata\_core\anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
C:\Users\natal\Desktop\UBC\OG_CPM\code\.venv\lib\site-packages\anndata\_core\anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
C:\Users\natal\Desktop\UBC\OG_CPM\code\.venv\lib\site-packages\anndata\_core\anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
C:\Users\natal\Desktop\UBC\OG_CPM\code\.venv\lib\site-packages\anndata\_core\anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
C:\Users\natal\Desktop\UBC\OG_CPM\code\.venv\lib\site-packages\anndata\_core\anndata.py:1758

Raw combined: (29870, 24356)


In [33]:
meta = pd.read_csv(METADATA_PATH, index_col=0)
print(f"Mantri's metadata: {meta.shape}")
print(f'AnnData barcode example: {adata.obs_names[0]}')
print(f'Metadata barcode example: {meta.index[0]}')

common = adata.obs_names.intersection(meta.index)
print(f'Overlap: {len(common)}')
assert len(common) == len(meta), (
    'Barcode format mismatch. Inspect the two examples printed above '
    'and adjust the obs_names reformatting in step 1.'
)

adata = adata[meta.index].copy()

Mantri's metadata: (22315, 17)
AnnData barcode example: D4_AAACCCAAGCTAAACA
Metadata barcode example: D4_AAACCCAAGCTAAACA
Overlap: 22315


In [34]:
sc.pp.filter_genes(adata, min_cells=3)
print(f'After gene filter: {adata.shape}')

After gene filter: (22315, 16390)


In [35]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [36]:
df = adata.to_df()
df['celltype'] = meta.loc[adata.obs_names, 'celltypes.0.5'].values
df.to_csv(OUTPUT_CSV, index=False)
print(f'Wrote {OUTPUT_CSV}: {adata.shape}')
print(f'Last col: {df.columns[-1]}, example values: {df.iloc[:5, -1].tolist()}')

Wrote chicken_cardio/chicken_cardio.csv: (22315, 16390)
Last col: celltype, example values: ['Cardiomyocytes-1', 'Cardiomyocytes-2', 'Immature myocardial cells', 'Cardiomyocytes-2', 'Cardiomyocytes-2']


In [37]:
import pandas as pd
df = pd.read_csv('chicken_cardio/chicken_cardio.csv', nrows=5)
print('Last column name:', df.columns[-1])
print('First 5 values:', df.iloc[:, -1].tolist())

labels = pd.read_csv('chicken_cardio/chicken_cardio.csv', usecols=[df.columns[-1]])
print('Unique labels:', labels.iloc[:, 0].unique())

Last column name: celltype
First 5 values: ['Cardiomyocytes-1', 'Cardiomyocytes-2', 'Immature myocardial cells', 'Cardiomyocytes-2', 'Cardiomyocytes-2']
Unique labels: ['Cardiomyocytes-1' 'Cardiomyocytes-2' 'Immature myocardial cells'
 'Valve cells' 'Macrophages' 'Fibroblast cells' 'Erythrocytes'
 'Endocardial cells' 'MT-enriched cardiomyocytes' 'Epi-epithelial cells'
 'Vascular endothelial cells' 'TMSB4X high cells' 'Epi-mesenchymal cells'
 'Dendritic cells' 'Mural cells']
